# Annotation conversion and pretrained segmentation demo

Convert **one real CytoNuke image** and **one real NuInsSeg image/mask pair**, then run pretrained **HoVer-NeXt** and **Cellpose 3 nuclei (non-SAM)** on two matched 256 × 256 patches. **UNI** is an optional patch encoder, not an instance segmenter; it is explicitly skipped when its authorized checkpoint is absent.

Conversion assertions establish file and coordinate mechanics. These two illustrative patches do not establish segmentation accuracy, clinical utility or comparative model performance.

## Setup and bounds

Use the project's locked `.venv` kernel from this checkout or `notebooks/`; set `PROJECT` explicitly when launching elsewhere. Public datasets must already exist locally. No data or weights are downloaded here.

Review the paths below. NuInsSeg's modified instance mask, RGB image and ambiguous-area mask must describe the same sample. `max_images=1` bounds conversion, but the COCO JSON is parsed in full, so JSON bytes have a separate bound.

See the [annotation conversion guide](../docs/annotation-conversion.md) for source and licence details. The source notebook is output-free; fresh figures and receipts go under ignored `outputs/annotation_segmentation_demo/` directories.

Pretrained inference uses the approved `pretrained` extra and the separate locked `environments/cellpose` environment. CPU threads are limited to one. Fresh automated execution uses a 120-second limit per code cell; each model/patch call has its own cell. Cellpose additionally enforces a 120-second worker timeout. Interactive HoVer-NeXt calls do not have a built-in interrupt timer.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import os
import json
import sys
import time
import uuid

for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[variable] = '1'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from shapely.geometry import shape

PROJECT = None  # Optional: Path(...) to this checkout.
if PROJECT is None:
    PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file()
                    and (p / 'src/annotations.py').is_file()), None)
    if PROJECT is None:
        raise FileNotFoundError('Open from this project or set PROJECT explicitly.')
PROJECT = Path(PROJECT).expanduser().resolve(strict=True)
sys.path.insert(0, str(PROJECT))
from src.annotations import convert_annotations, read_mask

COCO_PATH = PROJECT / 'data/cytonuke/extracted/coco.json'
COCO_IMAGE_ROOT = COCO_PATH.parent / 'images'
NUINSSEG_ROOT = PROJECT / 'data/nuinsseg/extracted/human bladder'
NUINSSEG_SAMPLE = 'human_bladder_01'
NUINSSEG_RGB = NUINSSEG_ROOT / 'tissue images' / f'{NUINSSEG_SAMPLE}.png'
NUINSSEG_MASK = NUINSSEG_ROOT / 'label masks modify' / f'{NUINSSEG_SAMPLE}.tif'
NUINSSEG_AMBIGUOUS = NUINSSEG_ROOT / 'vague areas/mask binary' / f'{NUINSSEG_SAMPLE}.png'
MAX_IMAGES = 1
MAX_PIXELS = 512 * 512
MAX_COCO_JSON_BYTES = 64_000_000
COCO_OVERLAP = 'last'  # Later source annotations win contested raster pixels.
assert MAX_IMAGES == 1 and 0 < MAX_PIXELS <= 1_048_576
for source in (COCO_PATH, NUINSSEG_RGB, NUINSSEG_MASK, NUINSSEG_AMBIGUOUS):
    assert source.is_file(), f'Local input missing: {source}'
assert COCO_PATH.stat().st_size <= MAX_COCO_JSON_BYTES
assert NUINSSEG_RGB.stem == NUINSSEG_MASK.stem == NUINSSEG_AMBIGUOUS.stem
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = PROJECT / 'outputs/annotation_segmentation_demo' / run_id
OUT.mkdir(parents=True, exist_ok=False)
started = time.monotonic()
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print(f'Python: {sys.executable}')
print(f'Outputs: {OUT}')


## 1. Convert one CytoNuke COCO image

CytoNuke category **0 is NUCLEUS**, not background; category 1 is CELL. Each category gets a separate TIFF. Zero is raster background; positive values are instance IDs. `.labels.json` retains original annotation IDs and categories.

Nuclei and cells overlap across categories, and within-category overlaps also occur. The explicit `last` policy resolves contested raster pixels in source order; it does not establish a biological boundary. GeoJSON retains original polygons and overlaps.

Coordinates are **local image pixels**: x = column, y = row downward, origin at the upper-left edge. A pixel spans `[x,x+1] × [y,y+1]`; rasterization tests pixel centres. No slide origin, micron scale, geographic CRS or WSI registration is inferred from filenames.

In [ ]:
cyto_dir = OUT / 'cytonuke'
cyto_manifest = convert_annotations(COCO_PATH, cyto_dir, input_format='coco',
    output_format='both', max_images=MAX_IMAGES, max_pixels=MAX_PIXELS,
    overlap=COCO_OVERLAP)
assert cyto_manifest['status'] == 'complete' and len(cyto_manifest['samples']) == 1
cyto_sample = cyto_manifest['samples'][0]
cyto_rgb_path = (COCO_IMAGE_ROOT / cyto_sample['source_image']).resolve(strict=True)
assert cyto_rgb_path.is_relative_to(COCO_IMAGE_ROOT.resolve())
with Image.open(cyto_rgb_path) as image:
    assert image.width * image.height <= MAX_PIXELS
    cyto_rgb = np.asarray(image.convert('RGB'))
assert cyto_rgb.shape[:2] == (cyto_sample['height'], cyto_sample['width'])
cyto_masks = {item['category_id']: read_mask(cyto_dir / item['path'], MAX_PIXELS)
              for item in cyto_sample['masks']}
cyto_geojson = json.loads((cyto_dir / next(p for p in cyto_sample['outputs']
                                        if p.endswith('.geojson'))).read_text())
cyto_labels = json.loads((cyto_dir / next(p for p in cyto_sample['outputs']
                                       if p.endswith('.labels.json'))).read_text())
assert 0 in cyto_masks and any(row['category_id'] == 0 and row['category_name'] == 'NUCLEUS'
                              for row in cyto_labels)
assert all(labels.dtype == np.uint32 and labels.shape == cyto_rgb.shape[:2]
           for labels in cyto_masks.values())
display(pd.DataFrame(cyto_sample['masks'])[
    ['category_id', 'instances', 'overlap_pixels', 'foreground_pixels',
     'clipped_instance_ids', 'unrepresented_instance_ids']])
print('Selected image:', cyto_sample['source_image'])


In [ ]:
def draw_annotation_contours(ax, rgb, features, *, category_id=None, color='lime'):
    """Plot saved pixel-edge polygons without converting or changing labels."""
    height, width = rgb.shape[:2]
    ax.imshow(rgb, extent=(0, width, height, 0))
    for feature in features:
        if category_id is not None and feature['properties'].get('category_id') != category_id:
            continue
        geometry = shape(feature['geometry'])
        polygons = [geometry] if geometry.geom_type == 'Polygon' else list(geometry.geoms)
        for polygon in polygons:
            ax.plot(*polygon.exterior.xy, color=color, linewidth=0.7)
            for ring in polygon.interiors:
                ax.plot(*ring.xy, color=color, linewidth=0.7, linestyle='--')
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect('equal')
    ax.axis('off')

fig, axes = plt.subplots(2, 3, figsize=(11, 7), layout='constrained')
axes[0, 0].imshow(cyto_rgb)
axes[0, 0].set_title('CytoNuke RGB')
for column, (category, name) in enumerate([(0, 'NUCLEUS'), (1, 'CELL')], 1):
    axes[0, column].imshow(np.ma.masked_equal(cyto_masks[category], 0), cmap='turbo', interpolation='nearest')
    axes[0, column].set_title(f'{name}: positive instance IDs')
    draw_annotation_contours(axes[1, column], cyto_rgb, cyto_geojson['features'], category_id=category)
    axes[1, column].set_title(f'{name}: original polygon boundaries')
axes[1, 0].text(0.05, 0.5, 'Colours identify instances, not scores.', transform=axes[1, 0].transAxes)
for ax in axes.flat:
    ax.axis('off')
fig.savefig(OUT / '01_cytonuke_annotations.png')
plt.show()


## 2. Preserve the modified NuInsSeg instance mask

Use **`label masks modify`**, the authors' recommended evaluation mask. Raw `label masks` can contain extra numerical values formed at overlaps; these are not automatically additional nuclei.

Positive integer IDs—including gaps—are preserved exactly. A binary foreground mask would be one instance, not automatically split into nuclei. Keep the paired ambiguous-area mask separate: conversion does not silently turn ambiguity into ordinary foreground/background or choose an evaluation ignore policy.

The views compare source RGB, author-provided instances, converted boundaries and the separate ambiguous mask. These are reference annotations, not predictions.

In [ ]:
with Image.open(NUINSSEG_RGB) as image:
    assert image.width * image.height <= MAX_PIXELS
    nu_rgb = np.asarray(image.convert('RGB'))
nu_source_labels = read_mask(NUINSSEG_MASK, MAX_PIXELS)
nu_ambiguous_labels = read_mask(NUINSSEG_AMBIGUOUS, MAX_PIXELS)
assert set(np.unique(nu_ambiguous_labels)).issubset({0, 255})
nu_ambiguous = nu_ambiguous_labels > 0
assert nu_rgb.shape[:2] == nu_source_labels.shape == nu_ambiguous.shape
nu_dir = OUT / 'nuinsseg'
nu_manifest = convert_annotations(NUINSSEG_MASK, nu_dir, input_format='mask',
    output_format='both', max_images=MAX_IMAGES, max_pixels=MAX_PIXELS)
assert nu_manifest['status'] == 'complete' and len(nu_manifest['samples']) == 1
nu_sample = nu_manifest['samples'][0]
nu_labels = read_mask(nu_dir / nu_sample['masks'][0]['path'], MAX_PIXELS)
assert np.array_equal(nu_labels, nu_source_labels), 'Conversion changed integer labels.'
nu_geojson = json.loads((nu_dir / next(p for p in nu_sample['outputs'] if p.endswith('.geojson'))).read_text())
nu_label_table = json.loads((nu_dir / next(p for p in nu_sample['outputs'] if p.endswith('.labels.json'))).read_text())
nu_ids = set(int(v) for v in np.unique(nu_source_labels) if v)
assert {item['instance_id'] for item in nu_label_table} == nu_ids
assert {item['source_label'] for item in nu_label_table} == nu_ids
print(f'{NUINSSEG_SAMPLE}: {len(nu_ids)} positive IDs; {int(nu_ambiguous.sum()):,} ambiguous pixels kept separate.')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10), layout='constrained')
axes[0, 0].imshow(nu_rgb)
axes[0, 0].set_title('NuInsSeg original RGB')
axes[0, 1].imshow(np.ma.masked_equal(nu_labels, 0), cmap='turbo', interpolation='nearest')
axes[0, 1].set_title('Modified mask: unchanged integer IDs')
draw_annotation_contours(axes[1, 0], nu_rgb, nu_geojson['features'])
axes[1, 0].set_title('Converted pixel-edge boundaries')
axes[1, 1].imshow(nu_ambiguous, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[1, 1].set_title('Author-provided ambiguous regions')
for ax in axes.flat:
    ax.axis('off')
fig.savefig(OUT / '02_nuinsseg_annotations.png')
plt.show()


## 3. Check the conversions and record provenance

Mask-to-TIFF conversion must preserve the complete integer array, not just its maximum or foreground area. COCO categories and annotation IDs remain explicit; category 0 must not disappear as background. GeoJSON declares image pixels rather than geographic coordinates.

Source paths and hashes identify exact files. Source images/masks remain in their canonical locations; only derived conversion outputs and figures are written here.

In [ ]:
for collection, rgb in [(cyto_geojson, cyto_rgb), (nu_geojson, nu_rgb)]:
    assert collection['coordinate_system'] == 'image-pixels' and collection['origin'] == 'top-left'
    assert (collection['width'], collection['height']) == (rgb.shape[1], rgb.shape[0])
assert {feature['id'] for feature in cyto_geojson['features']} == {row['instance_id'] for row in cyto_labels}
assert {feature['id'] for feature in nu_geojson['features']} == nu_ids
for category, labels in cyto_masks.items():
    allowed = {row['instance_id'] for row in cyto_labels if row['category_id'] == category}
    assert set(int(v) for v in np.unique(labels) if v) <= allowed
    assert all(row['instance_id'] > 0 for row in cyto_labels if row['category_id'] == category)
source_files = {'coco_json': COCO_PATH, 'cytonuke_rgb': cyto_rgb_path,
    'nuinsseg_rgb': NUINSSEG_RGB, 'nuinsseg_modified_mask': NUINSSEG_MASK,
    'nuinsseg_ambiguous_mask': NUINSSEG_AMBIGUOUS}
source_receipts = {}
for name, path in source_files.items():
    with path.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    source_receipts[name] = {'path': str(path), 'bytes': path.stat().st_size, 'sha256': digest}
annotation_receipt = {'status': 'complete', 'sources': source_receipts,
    'max_images_per_dataset': MAX_IMAGES, 'max_pixels_per_image': MAX_PIXELS,
    'max_coco_json_bytes': MAX_COCO_JSON_BYTES, 'coco_overlap_policy': COCO_OVERLAP,
    'cytonuke_image_id': cyto_sample['image_id'], 'cytonuke_masks': cyto_sample['masks'],
    'nuinsseg_sample': NUINSSEG_SAMPLE, 'nuinsseg_instances': len(nu_ids),
    'nuinsseg_integer_array_preserved': True, 'nuinsseg_ambiguous_pixels': int(nu_ambiguous.sum()),
    'ambiguous_region_policy': 'preserved separately; no evaluation policy applied',
    'coordinates': 'local pixel edges, x=column, y=row-down; no WSI origin or scale inferred',
    'annotation_source_sha256': hashlib.sha256((PROJECT / 'src/annotations.py').read_bytes()).hexdigest(),
    'versions': {name: version(name) for name in ['numpy', 'Pillow', 'shapely', 'tifffile']}}
(OUT / 'annotation_receipt.json').write_text(json.dumps(annotation_receipt, indent=2))
print(f'Annotation checks passed in {time.monotonic() - started:.1f} seconds. Outputs: {OUT}')


## 4. Prepare two matched inference patches

CytoNuke uses its complete 256 × 256 image and **category 0 nuclei** only. NuInsSeg uses a deterministic centre crop of the 512 × 512 original: `(x, y, width, height) = (128, 128, 256, 256)`. RGB, reference IDs and ambiguity are cropped identically, without resizing or label renumbering. Some instances may meet the crop boundary.

Model outputs use patch-local coordinates. Add the recorded crop origin to map them back to the source image. Physical pixel size is not verified from these PNGs; no magnification equivalence is asserted. CytoNuke supplies no separate ambiguity mask here, so its pixels are all included in the illustrative comparison; that does not imply that every annotation is certain.

In [ ]:
PATCH_SIZE = 256
MAX_MODEL_PIXELS = PATCH_SIZE ** 2
assert cyto_rgb.shape == (PATCH_SIZE, PATCH_SIZE, 3)
assert nu_rgb.shape == (512, 512, 3)
nu_x = nu_y = 128
nu_crop = np.s_[nu_y:nu_y + PATCH_SIZE, nu_x:nu_x + PATCH_SIZE]
patches = {
    'cytonuke': {'rgb': cyto_rgb.copy(), 'reference': cyto_masks[0].copy(),
        'ambiguous': np.zeros(cyto_rgb.shape[:2], dtype=bool),
        'source': str(cyto_rgb_path), 'source_size_wh': [256, 256],
        'crop_xywh': [0, 0, 256, 256], 'reference_category': 'NUCLEUS (COCO category 0)',
        'ambiguity_policy': 'No separate supplied mask; include all pixels'},
    'nuinsseg': {'rgb': nu_rgb[nu_crop].copy(), 'reference': nu_labels[nu_crop].copy(),
        'ambiguous': nu_ambiguous[nu_crop].copy(),
        'source': str(NUINSSEG_RGB), 'source_size_wh': [512, 512],
        'crop_xywh': [nu_x, nu_y, 256, 256], 'reference_category': 'modified nuclei labels',
        'ambiguity_policy': 'Exclude author-provided ambiguous pixels'},
}
assert len(patches) == 2
patch_provenance = {}
for name, patch in patches.items():
    assert patch['rgb'].shape == (256, 256, 3) and patch['rgb'].dtype == np.uint8
    assert patch['reference'].shape == patch['ambiguous'].shape == (256, 256)
    assert patch['reference'].dtype == np.uint32 and patch['ambiguous'].dtype == bool
    patch_provenance[name] = {key: value for key, value in patch.items()
                              if key not in {'rgb', 'reference', 'ambiguous'}}
    patch_provenance[name].update({
        'rgb_sha256': hashlib.sha256(patch['rgb'].tobytes()).hexdigest(),
        'reference_sha256': hashlib.sha256(patch['reference'].tobytes()).hexdigest(),
        'ambiguous_sha256': hashlib.sha256(patch['ambiguous'].tobytes()).hexdigest(),
        'reference_ids': [int(v) for v in np.unique(patch['reference']) if v],
        'ambiguous_pixels': int(patch['ambiguous'].sum()), 'input_mpp_verified': False})
fig, axes = plt.subplots(2, 3, figsize=(10, 7), layout='constrained')
for row, (name, patch) in enumerate(patches.items()):
    axes[row, 0].imshow(patch['rgb'])
    axes[row, 0].set_title(f'{name}: exact model RGB')
    axes[row, 1].imshow(np.ma.masked_equal(patch['reference'], 0), cmap='turbo', interpolation='nearest')
    axes[row, 1].set_title('Reference nuclei IDs')
    axes[row, 2].imshow(patch['ambiguous'], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    axes[row, 2].set_title('Excluded ambiguous pixels')
for ax in axes.flat:
    ax.axis('off')
fig.savefig(OUT / '03_matched_model_inputs.png')
plt.show()
(OUT / 'patch_provenance.json').write_text(json.dumps(patch_provenance, indent=2))


## 5. Load pinned pretrained models

The loaders verify local checkpoint hashes and never silently download weights or fall back to random parameters. HoVer-NeXt uses the official **PanNuke ConvNeXtV2 Tiny fold-1** checkpoint, one unaugmented direct patch, and upstream postprocessing thresholds. It receives RGB divided by 255; no stain normalization, resizing, WSI padding or test-time augmentation is applied. Its PanNuke scale assumption is 0.25 µm/pixel, which is **not established for these PNG inputs**.

Cellpose uses the pretrained **`nucleitorch_0`** nuclei checkpoint in its separate environment, grayscale channels `(0, 0)`, `diameter=17` input pixels, `invert=False` and its recorded default percentile normalization. These are declared settings, not tuned to these annotations. Predictions from both models are compared as nuclei foreground, without mapping HoVer-NeXt's PanNuke classes to reference cell types.

UNI is the original 1024-feature encoder (not UNI2). Set `UNI_CHECKPOINT` only to a locally authorized original-UNI checkpoint. Without it, the notebook records `not_executed`; no embedding or segmentation result is invented. The project setup access probe returned HTTP 403 for gated weights.

In [ ]:
import torch
import tifffile
from src.pretrained import load_uni, embed_uni, load_hovernext, predict_hovernext, run_cellpose_patch

torch.set_num_threads(1)
if torch.get_num_interop_threads() != 1:
    torch.set_num_interop_threads(1)  # Restart the kernel before changing thread policy.
DEVICE = 'cpu'
MODEL_TIMEOUT_SECONDS = 120
HOVERNEXT_CHECKPOINT = PROJECT / 'data/models/hovernext/pannuke_convnextv2_tiny_1'
CELLPOSE_CHECKPOINT = PROJECT / 'data/models/cellpose/nucleitorch_0'
CELLPOSE_SHA256 = '89ca45e4a45048d5010d29621466b1274b91b3dcb9714dce9f9e90a9a8671303'
UNI_CHECKPOINT = None  # Optional Path to an authorized original-UNI checkpoint.
CELLPOSE_DIAMETER = 17.0
CELLPOSE_INVERT = False
CELLPOSE_CHANNELS = (0, 0)
assert HOVERNEXT_CHECKPOINT.is_dir() and CELLPOSE_CHECKPOINT.is_file()
hovernext = load_hovernext(HOVERNEXT_CHECKPOINT, device=DEVICE)
uni = None
uni_receipt = {'status': 'not_executed', 'reason': 'No authorized local UNI checkpoint supplied',
               'model': 'original UNI', 'features': 1024}
if UNI_CHECKPOINT is not None:
    uni = load_uni(UNI_CHECKPOINT, device=DEVICE)
    uni_receipt = {'status': 'loaded', 'metadata': uni['metadata'], 'patches': {}}
predictions = {name: {} for name in patches}
uni_embeddings = {}
print('HoVer-NeXt loaded on CPU; Cellpose will use its isolated CPU worker.')
print('UNI:', uni_receipt['status'])


In [ ]:
def save_prediction(name, model, result, elapsed_seconds):
    """Retain derived predictions separately from ground truth and verify saved labels."""
    labels = result['instance_mask']
    assert labels.shape == (256, 256) and labels.dtype == np.uint32
    assert hashlib.sha256(patches[name]['rgb'].tobytes()).hexdigest() == patch_provenance[name]['rgb_sha256']
    directory = OUT / 'predictions' / model / name
    directory.mkdir(parents=True, exist_ok=False)
    tifffile.imwrite(directory / 'instances.tif', labels, photometric='minisblack',
                     compression='deflate', metadata=None)
    assert np.array_equal(tifffile.imread(directory / 'instances.tif'), labels)
    with (directory / 'instances.tif').open('rb') as stream:
        mask_file_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
    metadata = {**result['metadata'], 'elapsed_notebook_seconds': elapsed_seconds,
                'patch': patch_provenance[name], 'mask': 'instances.tif',
                'mask_sha256': mask_file_sha256,
                'mask_array_sha256': hashlib.sha256(labels.tobytes()).hexdigest(),
                'mask_shape': list(labels.shape), 'mask_dtype': str(labels.dtype)}
    (directory / 'metadata.json').write_text(json.dumps(metadata, indent=2, allow_nan=False))
    if 'classes' in result:
        assert set(np.unique(labels)) - {0} == result['classes'].keys()
        assert all(np.isfinite(value).all() for value in result['probabilities'].values())
        (directory / 'classes.json').write_text(json.dumps(result['classes'], indent=2))
    predictions[name][model] = {'labels': labels, 'metadata': metadata}
    print(f'{model} / {name}: {len(np.unique(labels[labels > 0]))} predicted IDs, {elapsed_seconds:.2f}s')


In [ ]:
call_started = time.monotonic()
hover_result = predict_hovernext(patches['cytonuke']['rgb'], hovernext, max_pixels=MAX_MODEL_PIXELS)
save_prediction('cytonuke', 'hovernext', hover_result, time.monotonic() - call_started)


In [ ]:
call_started = time.monotonic()
hover_result = predict_hovernext(patches['nuinsseg']['rgb'], hovernext, max_pixels=MAX_MODEL_PIXELS)
save_prediction('nuinsseg', 'hovernext', hover_result, time.monotonic() - call_started)


In [ ]:
call_started = time.monotonic()
cellpose_result = run_cellpose_patch(patches['cytonuke']['rgb'], OUT / 'cellpose_worker' / 'cytonuke',
    checkpoint=CELLPOSE_CHECKPOINT, checkpoint_sha256=CELLPOSE_SHA256,
    diameter=CELLPOSE_DIAMETER, invert=CELLPOSE_INVERT, channels=CELLPOSE_CHANNELS,
    max_pixels=MAX_MODEL_PIXELS, timeout_seconds=MODEL_TIMEOUT_SECONDS)
save_prediction('cytonuke', 'cellpose', cellpose_result, time.monotonic() - call_started)


In [ ]:
call_started = time.monotonic()
cellpose_result = run_cellpose_patch(patches['nuinsseg']['rgb'], OUT / 'cellpose_worker' / 'nuinsseg',
    checkpoint=CELLPOSE_CHECKPOINT, checkpoint_sha256=CELLPOSE_SHA256,
    diameter=CELLPOSE_DIAMETER, invert=CELLPOSE_INVERT, channels=CELLPOSE_CHANNELS,
    max_pixels=MAX_MODEL_PIXELS, timeout_seconds=MODEL_TIMEOUT_SECONDS)
save_prediction('nuinsseg', 'cellpose', cellpose_result, time.monotonic() - call_started)


In [ ]:
if uni is not None:
    call_started = time.monotonic()
    embedding = embed_uni(patches['cytonuke']['rgb'], uni, max_pixels=MAX_MODEL_PIXELS)
    assert embedding.shape == (1024,) and embedding.dtype == np.float32 and np.isfinite(embedding).all()
    uni_embeddings['cytonuke'] = embedding
    np.save(OUT / 'uni_cytonuke.npy', embedding, allow_pickle=False)
    uni_receipt['patches']['cytonuke'] = {'shape': [1024], 'dtype': 'float32',
        'elapsed_seconds': time.monotonic() - call_started}
else:
    print('UNI / cytonuke: not executed — authorized checkpoint unavailable.')


In [ ]:
if uni is not None:
    call_started = time.monotonic()
    embedding = embed_uni(patches['nuinsseg']['rgb'], uni, max_pixels=MAX_MODEL_PIXELS)
    assert embedding.shape == (1024,) and embedding.dtype == np.float32 and np.isfinite(embedding).all()
    uni_embeddings['nuinsseg'] = embedding
    np.save(OUT / 'uni_nuinsseg.npy', embedding, allow_pickle=False)
    uni_receipt['patches']['nuinsseg'] = {'shape': [1024], 'dtype': 'float32',
        'elapsed_seconds': time.monotonic() - call_started}
else:
    print('UNI / nuinsseg: not executed — authorized checkpoint unavailable.')


## 6. Inspect predictions and a bounded foreground comparison

The figures retain RGB, reference IDs and each model's predicted IDs separately. IDs and colours are arbitrary and do not correspond across masks. HoVer-NeXt class probabilities are not calibrated confidence estimates.

The table's **foreground Dice** is `2 × overlap / (reference foreground + predicted foreground)` over included pixels. NuInsSeg ambiguous pixels are excluded from both masks; CytoNuke uses nuclei category 0 only. An empty denominator is reported as unavailable. This collapses instances into foreground and cannot assess splits, merges or instance matching. The two selected images are not a held-out benchmark; unknown pixel calibration, crop-edge effects and dataset/model exposure limit interpretation.

In [ ]:
from skimage.segmentation import find_boundaries

comparison_rows = []
fig, axes = plt.subplots(2, 4, figsize=(13, 7), layout='constrained')
for row, (name, patch) in enumerate(patches.items()):
    axes[row, 0].imshow(patch['rgb'])
    axes[row, 0].set_title(f'{name}: RGB')
    axes[row, 1].imshow(np.ma.masked_equal(patch['reference'], 0), cmap='turbo', interpolation='nearest')
    axes[row, 1].set_title('Reference nuclei IDs')
    valid = ~patch['ambiguous']
    reference_foreground = (patch['reference'] > 0) & valid
    for column, model in enumerate(('hovernext', 'cellpose'), 2):
        labels = predictions[name][model]['labels']
        predicted_foreground = (labels > 0) & valid
        denominator = int(reference_foreground.sum() + predicted_foreground.sum())
        dice = (2 * int((reference_foreground & predicted_foreground).sum()) / denominator
                if denominator else None)
        comparison_rows.append({'patch': name, 'model': model,
            'reference_ids_in_crop': len(np.unique(patch['reference'][patch['reference'] > 0])),
            'predicted_ids_in_crop': len(np.unique(labels[labels > 0])),
            'included_pixels': int(valid.sum()), 'excluded_ambiguous_pixels': int(patch['ambiguous'].sum()),
            'foreground_dice_illustrative': dice})
        axes[row, column].imshow(np.ma.masked_equal(labels, 0), cmap='turbo', interpolation='nearest')
        axes[row, column].set_title(f'{model}: predicted IDs')
for ax in axes.flat:
    ax.axis('off')
fig.savefig(OUT / '04_predicted_instances.png')
plt.show()
comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(OUT / 'illustrative_foreground_comparison.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), layout='constrained')
for row, (name, patch) in enumerate(patches.items()):
    for column, model in enumerate(('hovernext', 'cellpose')):
        ax = axes[row, column]
        ax.imshow(patch['rgb'])
        for labels, color in [(patch['reference'], (0, 1, 0, 1)),
                              (predictions[name][model]['labels'], (1, 0, 1, 1))]:
            outline = np.zeros((256, 256, 4), dtype=float)
            outline[find_boundaries(labels, mode='inner') & ~patch['ambiguous']] = color
            ax.imshow(outline)
        ax.set_title(f'{name} / {model}')
        ax.axis('off')
fig.suptitle('Green: reference boundaries · Magenta: prediction boundaries\nAmbiguous pixels excluded from outlines and Dice')
fig.savefig(OUT / '05_reference_prediction_boundaries.png')
plt.show()


## 7. Record what actually ran

The final receipt records input crops/hashes, checkpoint metadata, model settings, saved label checks and UNI's actual execution status. Source files are checked again after inference. Prediction masks are uint32 TIFFs under `predictions/`; original source images and annotations remain in place. The receipt's completion status refers to the available segmentation workflow, not successful execution of unavailable gated models or validated model efficacy.

In [ ]:
for name, source in source_receipts.items():
    with Path(source['path']).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == source['sha256'], name
for name, patch in patches.items():
    for key in ('rgb', 'reference', 'ambiguous'):
        assert hashlib.sha256(patch[key].tobytes()).hexdigest() == patch_provenance[name][f'{key}_sha256']
    assert set(predictions[name]) == {'hovernext', 'cellpose'}
if uni is not None:
    assert set(uni_embeddings) == set(patches)
    uni_receipt['status'] = 'complete'
(OUT / 'uni_status.json').write_text(json.dumps(uni_receipt, indent=2))
receipt = {'status': 'complete', 'scope': 'bounded annotation conversion and available pretrained inference',
    'patches': patch_provenance, 'sources': source_receipts, 'source_files_unchanged': True,
    'models': {model: {name: predictions[name][model]['metadata'] for name in patches}
               for model in ('hovernext', 'cellpose')},
    'uni': uni_receipt, 'device': DEVICE, 'torch_threads': torch.get_num_threads(),
    'torch_interop_threads': torch.get_num_interop_threads(),
    'patches_per_model_maximum': 2, 'max_model_pixels': MAX_MODEL_PIXELS,
    'cellpose_worker_timeout_seconds': MODEL_TIMEOUT_SECONDS,
    'foreground_comparison': comparison_rows,
    'interpretation': 'Two illustrative patches; foreground comparison is not instance accuracy or a benchmark',
    'source_code_sha256': {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest()
        for name in ('src/annotations.py', 'src/pretrained.py', 'src/hovernext.py', 'src/cellpose_worker.py')},
    'elapsed_seconds': time.monotonic() - started}
(OUT / 'run_receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
print(f"Complete: 2 annotation examples, 4 segmentation calls; UNI {uni_receipt['status']}.")
print(f'Outputs: {OUT}')
